# DynamoDB 데이터 준비 - prepare_ddb

로컬 `conf/`와 `data/`를 DynamoDB 테이블 `gsretail-mlops-edu-hjsong`에 등록합니다.

## 테이블 키 구조
```
experiment_id (HASH key)   |  entity_type (RANGE key)
────────────────────────────────────────────────────────
EXP#{user_id}#{project}#{experiment}  |  META
EXP#{user_id}#{project}#{experiment}  |  CONF
EXP#{user_id}#{project}#{experiment}  |  DATA#train
EXP#{user_id}#{project}#{experiment}  |  DATA#validation
EXP#{user_id}#{project}#{experiment}  |  DATA#test
```

## 1. 라이브러리 및 기본 설정

In [ ]:
import sys, os, yaml, json, base64, hashlib, boto3
from pathlib import Path
from decimal import Decimal
from datetime import datetime

BASE_DIR = Path.cwd()                                      # ddb/
CONF_DIR = BASE_DIR.parent / 'prepare_input' / 'conf'     # prepare_input/conf/
DATA_DIR = BASE_DIR.parent / 'prepare_input' / 'data'     # prepare_input/data/

print(f'CONF_DIR: {CONF_DIR}')
print(f'DATA_DIR: {DATA_DIR}')

## 2. 설정 파일 로드

In [ ]:
def load_yaml(p):
    with open(p, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

env_config   = load_yaml(CONF_DIR / 'env.yml')
meta_config  = load_yaml(CONF_DIR / 'meta.yml')
model_config = load_yaml(CONF_DIR / 'model.yml')

print('설정 로드 완료')
print(f'  env:        {env_config["env"]}')
print(f'  region:     {env_config["region"]}')
print(f'  user_id:    {meta_config["user_id"]}')
print(f'  project:    {meta_config["project"]}')
print(f'  experiment: {meta_config["experiment"]}')

## 3. DynamoDB 연결 및 키 구성

In [ ]:
# 실험 식별 정보
USER_ID    = meta_config['user_id']
PROJECT    = meta_config['project']
EXPERIMENT = meta_config['experiment']
VERSION    = meta_config['version']
ENV        = env_config['env']
REGION     = env_config['region']

TABLE_NAME = 'gsretail-mlops-edu-hjsong'

# DynamoDB 리소스 초기화
ddb   = boto3.resource('dynamodb', region_name=REGION)
table = ddb.Table(TABLE_NAME)

# experiment_id 키 생성 규칙: EXP#{user_id}#{project}#{experiment}
EXP_PK = f'EXP#{USER_ID}#{PROJECT}#{EXPERIMENT}'

print(f'Table        : {TABLE_NAME}')
print(f'experiment_id: {EXP_PK}')

## 4. DynamoDB 타입 변환 헬퍼

DynamoDB는 Python `float` 타입을 직접 저장하지 못합니다.
`Decimal`로 변환해야 합니다.

In [ ]:
def to_ddb(obj):
    """float → Decimal 재귀 변환 (DynamoDB 저장 전 필수)"""
    if isinstance(obj, float):
        return Decimal(str(obj))
    elif isinstance(obj, bool):
        return obj
    elif isinstance(obj, dict):
        return {k: to_ddb(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_ddb(v) for v in obj]
    return obj

print('to_ddb 헬퍼 정의 완료')

## 5. 테이블 확인 / 생성

In [ ]:
client = boto3.client('dynamodb', region_name=REGION)
try:
    client.describe_table(TableName=TABLE_NAME)
    print(f'테이블 [{TABLE_NAME}] 존재 확인')
except client.exceptions.ResourceNotFoundException:
    print(f'테이블 [{TABLE_NAME}] 생성 중...')
    client.create_table(
        TableName=TABLE_NAME,
        KeySchema=[
            {'AttributeName': 'experiment_id', 'KeyType': 'HASH'},
            {'AttributeName': 'entity_type',   'KeyType': 'RANGE'},
        ],
        AttributeDefinitions=[
            {'AttributeName': 'experiment_id', 'AttributeType': 'S'},
            {'AttributeName': 'entity_type',   'AttributeType': 'S'},
        ],
        BillingMode='PAY_PER_REQUEST',
    )
    client.get_waiter('table_exists').wait(TableName=TABLE_NAME)
    print(f'테이블 [{TABLE_NAME}] 생성 완료')

## 6. META 아이템 저장

실험 식별 정보를 저장합니다.
```
experiment_id = EXP#hjsong#titanic-...#tuned-v1
entity_type   = META
```

In [ ]:
now = datetime.utcnow().isoformat()

table.put_item(Item={
    'experiment_id': EXP_PK,      # HASH key
    'entity_type':   'META',       # RANGE key
    'user_id':    USER_ID,
    'project':    PROJECT,
    'experiment': EXPERIMENT,
    'version':    VERSION,
    'env':        ENV,
    'region':     REGION,
    'created_at': now,
    'status':     'active',
})
print(f'[저장] META  →  experiment_id={EXP_PK}, entity_type=META')

## 7. CONF 아이템 저장

env/meta/model YAML 3개를 DynamoDB Map 타입으로 저장합니다.
```
entity_type = CONF
```

In [ ]:
table.put_item(Item=to_ddb({   # YAML 안에 float 값이 있어 to_ddb 필요
    'experiment_id': EXP_PK,
    'entity_type':   'CONF',
    'env_yml':       env_config,    # dict → DynamoDB Map
    'meta_yml':      meta_config,
    'model_yml':     model_config,
    'uploaded_at':   now,
    'uploaded_by':   USER_ID,
}))
print(f'[저장] CONF  →  env_yml / meta_yml / model_yml 포함')

## 8. DATA 아이템 저장 (CSV → base64)

CSV 파일을 그대로 `base64`로 인코딩해서 저장합니다.
나중에 `base64.b64decode()` → `pd.read_csv(BytesIO(...))` 로 복원합니다.
```
entity_type = DATA#train / DATA#validation / DATA#test
```

In [ ]:
for split in ['train', 'validation', 'test']:
    csv_path  = DATA_DIR / f'{split}.csv'
    csv_bytes = csv_path.read_bytes()

    # CSV bytes → base64 문자열
    csv_b64   = base64.b64encode(csv_bytes).decode('utf-8')
    checksum  = 'sha256:' + hashlib.sha256(csv_bytes).hexdigest()[:16] + '...'
    row_count = sum(1 for _ in open(csv_path)) - 1

    table.put_item(Item={
        'experiment_id': EXP_PK,
        'entity_type':   f'DATA#{split}',   # DATA#train / DATA#validation / DATA#test
        'split':         split,
        'version':       VERSION,
        'row_count':     row_count,
        'size_bytes':    len(csv_bytes),
        'checksum':      checksum,
        'csv_b64':       csv_b64,           # base64 인코딩된 CSV 원본
        'uploaded_at':   now,
    })
    b64_kb = len(csv_b64) / 1024
    print(f'[저장] DATA#{split:<12} {row_count}rows  {len(csv_bytes)/1024:.1f}KB → base64 {b64_kb:.1f}KB')

print('\n데이터셋 등록 완료')

## 9. 검증

In [ ]:
print('DynamoDB 업로드 검증')
print('=' * 60)
for et in ['META', 'CONF', 'DATA#train', 'DATA#validation', 'DATA#test']:
    resp   = table.get_item(Key={'experiment_id': EXP_PK, 'entity_type': et})
    exists = 'Item' in resp
    print(f'  {"OK" if exists else "MISSING"}: entity_type={et}')
print('=' * 60)

## 10. 최종 요약

In [ ]:
print(f'Table        : {TABLE_NAME}')
print(f'experiment_id: {EXP_PK}')
print()
print('저장된 entity_type:')
for et in ['META', 'CONF', 'DATA#train', 'DATA#validation', 'DATA#test']:
    print(f'  - {et}')
print()
print('다음 단계: ddb/modeling_ddb.ipynb 실행')